# 👗 GRB_Technology — Virtual Clothing Try-On

This notebook sets up and runs the full virtual try-on system.

### ✅ Before you start:
1. Go to **Runtime → Change runtime type**
2. Select **GPU (T4)**
3. Click **Save**, then run each cell below in order


In [ ]:
# ── Cell 1: Clone the GRB_Technology repo ──────────────────────────────────
!git clone https://github.com/PM2003/GRB_Technology.git
%cd GRB_Technology
!ls

In [ ]:
# ── Cell 2: Install all dependencies ───────────────────────────────────────
!pip install -q \
    torch torchvision \
    fastapi uvicorn python-multipart \
    flask requests \
    rembg \
    gradio \
    pillow \
    opencv-python-headless \
    scipy \
    gdown

print('✅ All packages installed!')

In [ ]:
# ── Cell 3: Download pretrained ViTON model weights ────────────────────────
import os
os.makedirs('checkpoints', exist_ok=True)

# ViTON GMM (Geometric Matching Module) weights
!gdown --id 1UWT6esQIU_d4tUm7cFode7RA7RFund5Ob -O checkpoints/gmm_final.pth

# ViTON TOM (Try-On Module) weights
!gdown --id 1T5_YDe9itMfe2XL9dv0HkRoVC7Iq67zs  -O checkpoints/tom_final.pth

# U2Net cloth segmentation weights
!gdown --id 1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ  -O checkpoints/cloth_segm_u2net.pth

print('✅ Weights downloaded!')
!ls -lh checkpoints/

In [ ]:
# ── Cell 4: Start the inference server in the background ───────────────────
import subprocess, time, sys

proc = subprocess.Popen(
    [sys.executable, '-m', 'server.api_server'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(6)

# Quick health check
import requests
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print('✅ Inference server is running:', r.json())
except Exception as e:
    print('⚠️  Server may still be starting:', e)

In [ ]:
# ── Cell 5: Launch Gradio web interface ────────────────────────────────────
# A public shareable URL will be printed below — open it in any browser!

import gradio as gr
import requests, io
from PIL import Image

def virtual_tryon(person_img: Image.Image, cloth_img: Image.Image):
    """Send images to the inference server and return the try-on result."""
    if person_img is None or cloth_img is None:
        return None, '⚠️ Please upload both images.'

    buf_p, buf_c = io.BytesIO(), io.BytesIO()
    person_img.save(buf_p, format='PNG')
    cloth_img.save(buf_c,  format='PNG')
    buf_p.seek(0); buf_c.seek(0)

    try:
        resp = requests.post(
            'http://localhost:8000/api/transform',
            files={
                'model': ('person.png', buf_p, 'image/png'),
                'cloth': ('cloth.png',  buf_c, 'image/png')
            },
            timeout=120
        )
        if resp.status_code == 200:
            result = Image.open(io.BytesIO(resp.content))
            return result, '✅ Done!'
        else:
            return None, f'❌ Server error {resp.status_code}: {resp.text[:200]}'
    except Exception as e:
        return None, f'❌ Error: {str(e)}'


with gr.Blocks(title='GRB_Technology Virtual Try-On', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 👗 GRB_Technology — Virtual Clothing Try-On')
    gr.Markdown('Upload a **full-body person photo** and a **clothing item** to see the try-on result.')

    with gr.Row():
        person_input = gr.Image(type='pil', label='📷 Your Photo (full-body)')
        cloth_input  = gr.Image(type='pil', label='👕 Clothing Item')
        result_out   = gr.Image(type='pil', label='✨ Try-On Result')

    status_out = gr.Textbox(label='Status', interactive=False)

    with gr.Row():
        run_btn   = gr.Button('✨ Try It On!',  variant='primary')
        clear_btn = gr.Button('🔄 Clear')

    run_btn.click(
        fn=virtual_tryon,
        inputs=[person_input, cloth_input],
        outputs=[result_out, status_out]
    )
    clear_btn.click(
        fn=lambda: (None, None, None, ''),
        outputs=[person_input, cloth_input, result_out, status_out]
    )

demo.launch(share=True)  # share=True gives you a public URL
